# Figure 2. DEC clustering performance and seismic-state representation

This notebook draws a manuscript figure for the first Results section. The figure combines all-event stability diagnostics with one representative DEC clustering example.

**Panel logic**

- **a** AE pretraining loss for all 44 events: evaluates whether the encoder learns stable spectrogram representations across the dataset.
- **b** WCSS elbow curves for all 44 events: documents event-specific cluster-number selection.
- **c** DEC fine-tuning loss for a representative event.
- **d** Embedded-space scatter for the representative event.
- **e** Original and reconstructed spectrograms for the minimum-RMS sample within each cluster.

## 1. Imports, paths and plotting style

The notebook avoids pandas/openpyxl and reads CSV/XLSX files using the Python standard library, plus `numpy` and `matplotlib` for analysis and plotting.

In [ ]:
from pathlib import Path
import csv
import re
import zipfile
import xml.etree.ElementTree as ET

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben")
EVENT = "event3"

AE_DIR = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\AE"
DEC_DIR = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\DEC_result"
K_DIR = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\Figure\optimal_cluster"
BEST_K_FILE = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\Figure\summary_best_k.xlsx"
OUT_DIR = PROJECT_ROOT / r"uilt_oneday_all\uilt_paper_figure\Fig2_cluster_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

AE_FILES = sorted(AE_DIR.glob("event*_stft_maxmin_Z_pretrain_log.csv"), key=lambda p: int(re.search(r"event(\d+)", p.name).group(1)))
K_FILES = sorted(K_DIR.glob("event*.xlsx"), key=lambda p: int(re.search(r"event(\d+)", p.stem).group(1)))
DEC_LOSS = DEC_DIR / "loss_csv" / f"{EVENT}training_loss.csv"
NPZ_FILE = DEC_DIR / "all_parement" / f"{EVENT}_all_parement.npz"

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 7.5,
    "axes.labelsize": 7.5,
    "axes.titlesize": 8,
    "xtick.labelsize": 6.8,
    "ytick.labelsize": 6.8,
    "legend.fontsize": 6.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "legend.frameon": False,
})

print("AE files:", len(AE_FILES))
print("K files:", len(K_FILES))
print("Representative event:", EVENT)
print("OUT_DIR:", OUT_DIR)

## 2. Helper functions

In [ ]:
def event_order(name):
    match = re.search(r"event(\d+)", str(name))
    return int(match.group(1)) if match else 10**9


def read_csv_numeric(path):
    with open(path, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = []
        for row in reader:
            clean = {}
            for key, value in row.items():
                try:
                    clean[key] = float(value)
                except Exception:
                    clean[key] = value
            rows.append(clean)
    return rows


def col_to_index(cell_ref):
    letters = re.match(r"([A-Z]+)", cell_ref).group(1)
    value = 0
    for char in letters:
        value = value * 26 + ord(char) - ord("A") + 1
    return value - 1


def text_from_si(si_node, ns):
    return "".join(node.text or "" for node in si_node.findall(".//m:t", ns))


def read_xlsx_sheet(path, sheet_name="Sheet1"):
    ns = {
        "m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with zipfile.ZipFile(path) as zf:
        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            shared = [text_from_si(si, ns) for si in root.findall("m:si", ns)]
        workbook = ET.fromstring(zf.read("xl/workbook.xml"))
        rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
        rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("rel:Relationship", ns)}
        target = None
        for sheet in workbook.findall(".//m:sheet", ns):
            if sheet.attrib["name"] == sheet_name:
                target = rel_map[sheet.attrib[f"{{{ns['r']}}}id"]]
                break
        target = target.lstrip("/")
        sheet_xml = target if target.startswith("xl/") else f"xl/{target}"
        root = ET.fromstring(zf.read(sheet_xml))

    rows = []
    for row in root.findall(".//m:sheetData/m:row", ns):
        values = []
        for cell in row.findall("m:c", ns):
            idx = col_to_index(cell.attrib["r"])
            while len(values) <= idx:
                values.append("")
            cell_type = cell.attrib.get("t")
            if cell_type == "inlineStr":
                values[idx] = "".join(node.text or "" for node in cell.findall(".//m:t", ns))
                continue
            value_node = cell.find("m:v", ns)
            if value_node is None:
                values[idx] = ""
                continue
            raw = value_node.text
            if cell_type == "s":
                values[idx] = shared[int(raw)]
            else:
                try:
                    values[idx] = float(raw)
                except Exception:
                    values[idx] = raw
        rows.append(values)
    header = [str(v) for v in rows[0]]
    return [dict(zip(header, (values + [""] * len(header))[:len(header)])) for values in rows[1:]]


def stack_curves(curves):
    max_len = max(len(curve) for curve in curves)
    arr = np.full((len(curves), max_len), np.nan, dtype=float)
    for i, curve in enumerate(curves):
        arr[i, :len(curve)] = curve
    return arr


def moving_average(values, window=15):
    values = np.asarray(values, dtype=float)
    if len(values) < window:
        return values
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="same")


def panel_label(ax, label):
    ax.text(-0.16, 1.14, label, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top", ha="left")


def minimum_rms_indices(labels, spectral):
    reps = {}
    rms_values = {}
    spec = np.squeeze(spectral).astype(float)
    rms = np.sqrt(np.mean(spec ** 2, axis=tuple(range(1, spec.ndim))))
    for cluster_id in np.unique(labels):
        cluster_id = int(cluster_id)
        idx = np.where(labels == cluster_id)[0]
        rep_idx = int(idx[np.argmin(rms[idx])])
        reps[cluster_id] = rep_idx
        rms_values[cluster_id] = float(rms[rep_idx])
    return reps, rms_values


def show_spectrogram(ax, spec, vmin=None, vmax=None, title=None, self_norm=False, cmap="magma", show_axes=False):
    spec = np.squeeze(spec).astype(float)
    if self_norm:
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-12)
    extent = [0, 60, 0, 50] if show_axes else None
    im = ax.imshow(spec, aspect="auto", origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, extent=extent, interpolation="nearest")
    if show_axes:
        ax.set_xticks([0, 20, 40, 60])
        ax.set_yticks([0, 25, 50])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.45)
            spine.set_color("0.35")
    else:
        ax.set_xticks([])
        ax.set_yticks([])
    if title:
        ax.set_title(title, pad=2)
    return im

## 3. Load all-event AE loss, all-event K curves, and representative-event DEC results

In [ ]:
# AE loss from all retained events
train_curves = []
val_curves = []
for path in AE_FILES:
    rows = read_csv_numeric(path)
    train_curves.append(np.array([row["loss"] for row in rows], dtype=float))
    val_curves.append(np.array([row["val_loss"] for row in rows], dtype=float))
train_arr = stack_curves(train_curves)
val_arr = stack_curves(val_curves)
ae_epoch = np.arange(train_arr.shape[1])

# WCSS curves and event-specific best K
best_k_rows = read_xlsx_sheet(BEST_K_FILE, "Sheet1")
best_k = {str(row["Event"]): int(float(row["WCSS"])) for row in best_k_rows}
k_curves = []
for path in K_FILES:
    event = path.stem
    rows = read_xlsx_sheet(path, "Sheet1")
    k = np.array([row["n_clusters"] for row in rows], dtype=float)
    wcss = np.array([row["WCSS"] for row in rows], dtype=float)
    k_curves.append((event, k, wcss))

# Representative event DEC data
npz = np.load(NPZ_FILE, allow_pickle=True)
spectral = np.asarray(npz["spectral"])
reconstructed = np.asarray(npz["respectral"])
features_2d = np.asarray(npz["redu"])
labels = np.asarray(npz["y_pred_last"]).astype(int)
distances = np.asarray(npz["distances"])
n_clusters = int(np.asarray(npz["n_clusters"]))
dec_rows = read_csv_numeric(DEC_LOSS)

print("AE train array:", train_arr.shape)
print("AE val array:", val_arr.shape)
print("K curves:", len(k_curves))
print("Representative event:", EVENT, "K=", n_clusters, "labels=", np.unique(labels))
print("spectral:", spectral.shape, "reconstructed:", reconstructed.shape)

## 4. Draw the figure

The output file is saved as PNG in `Fig2_cluster_results`.

In [ ]:
cluster_palette = plt.get_cmap("tab10")
representatives, representative_rms = minimum_rms_indices(labels, spectral)
ordered_clusters = sorted(representatives)

fig = plt.figure(figsize=(7.35, 9.55))
outer = fig.add_gridspec(3, 2, height_ratios=[0.95, 0.95, 1.62], hspace=0.55, wspace=0.38)
ax_a = fig.add_subplot(outer[0, 0])
ax_b = fig.add_subplot(outer[0, 1])
ax_c = fig.add_subplot(outer[1, 0])
ax_d = fig.add_subplot(outer[1, 1])
spec_grid = outer[2, :].subgridspec(2, len(ordered_clusters), hspace=0.22, wspace=0.12)
spec_axes = [fig.add_subplot(spec_grid[row, col]) for row in range(2) for col in range(len(ordered_clusters))]

# a. AE loss across all events
train_mean = np.nanmean(train_arr, axis=0)
train_std = np.nanstd(train_arr, axis=0)
val_mean = np.nanmean(val_arr, axis=0)
val_std = np.nanstd(val_arr, axis=0)
for curve in train_arr:
    ax_a.plot(ae_epoch, curve, color="#365C8D", alpha=0.08, linewidth=0.55)
for curve in val_arr:
    ax_a.plot(ae_epoch, curve, color="#B85C38", alpha=0.08, linewidth=0.55)
ax_a.fill_between(ae_epoch, train_mean - train_std, train_mean + train_std, color="#365C8D", alpha=0.12, linewidth=0)
ax_a.fill_between(ae_epoch, val_mean - val_std, val_mean + val_std, color="#B85C38", alpha=0.12, linewidth=0)
ax_a.plot(ae_epoch, train_mean, color="#365C8D", linewidth=1.3, label="Training mean")
ax_a.plot(ae_epoch, val_mean, color="#B85C38", linewidth=1.3, label="Validation mean")
ax_a.set_xlabel("Epoch")
ax_a.set_ylabel("MSE loss")
ax_a.set_title("AE loss, all events")
ax_a.grid(alpha=0.16)
ax_a.legend(loc="upper right", handlelength=1.1)
panel_label(ax_a, "a")

# b. WCSS curves across all events
for event, k, wcss in k_curves:
    y = wcss / wcss[0]
    ax_b.plot(k, y, color="0.68", alpha=0.45, linewidth=0.75)
    if event in best_k:
        chosen = best_k[event]
        if chosen in k:
            idx = np.where(k == chosen)[0][0]
            ax_b.scatter(k[idx], y[idx], marker="*", s=18, color="#D95F02", edgecolor="none", alpha=0.75, zorder=4)
# Highlight representative event
for event, k, wcss in k_curves:
    if event == EVENT:
        y = wcss / wcss[0]
        ax_b.plot(k, y, color="#1F4E79", linewidth=1.45)
        if EVENT in best_k:
            chosen = best_k[EVENT]
            idx = np.where(k == chosen)[0][0]
            ax_b.scatter(k[idx], y[idx], marker="*", s=85, color="#D95F02", edgecolor="black", linewidth=0.35, zorder=5)
        break
ax_b.set_xlabel("Number of clusters (K)")
ax_b.set_ylabel("Normalized WCSS")
ax_b.set_title("Elbow curves, all events")
ax_b.grid(alpha=0.16)
ax_b.text(0.05, 0.92, f"{EVENT}: K={best_k.get(EVENT, n_clusters)}", transform=ax_b.transAxes, fontsize=6.8, fontweight="bold", va="top")
panel_label(ax_b, "b")

# c. DEC loss for representative event
dec_step = np.arange(len(dec_rows))
total_loss = np.array([row["total_loss"] for row in dec_rows], dtype=float)
cluster_loss = np.array([row["cluster_loss"] for row in dec_rows], dtype=float)
reconstruction_loss = np.array([row["reconstruction_loss"] for row in dec_rows], dtype=float)
smooth_window = max(5, min(51, len(dec_step) // 35 * 2 + 1))
ax_c.plot(dec_step, moving_average(total_loss, smooth_window), color="#1F4E79", linewidth=1.2, label="Total")
ax_c.plot(dec_step, moving_average(cluster_loss, smooth_window), color="#D95F02", linewidth=1.2, label="Cluster")
ax_c.plot(dec_step, moving_average(reconstruction_loss, smooth_window), color="#5B8C5A", linewidth=1.2, label="Recon.")
ax_c.set_xlabel("Training step")
ax_c.set_ylabel("Loss")
ax_c.set_title(f"DEC loss, {EVENT}")
ax_c.grid(alpha=0.16)
ax_c.legend(loc="upper right", handlelength=1.1)
panel_label(ax_c, "c")

# d. embedded scatter for representative event
for cluster_id in ordered_clusters:
    mask = labels == cluster_id
    ax_d.scatter(features_2d[mask, 0], features_2d[mask, 1], s=4, alpha=0.52, color=cluster_palette(cluster_id % 10), label=f"Cluster {cluster_id}", rasterized=True)
for cluster_id, idx in representatives.items():
    ax_d.scatter(features_2d[idx, 0], features_2d[idx, 1], s=40, marker="*", color="black", edgecolor="white", linewidth=0.35, zorder=5)
ax_d.set_xlabel("Embedding dimension 1")
ax_d.set_ylabel("Embedding dimension 2")
ax_d.set_title(f"{EVENT}: embedded seismic states")
ax_d.grid(alpha=0.14)
ax_d.legend(ncol=2, loc="upper right", markerscale=2.0, columnspacing=0.8, handletextpad=0.2)
panel_label(ax_d, "d")

# e. original and reconstructed spectrograms of minimum-RMS samples in each cluster
vmin, vmax = 0.0, 1.0

for col, cluster_id in enumerate(ordered_clusters):
    idx = representatives[cluster_id]
    ax_orig = spec_axes[col]
    ax_rec = spec_axes[len(ordered_clusters) + col]
    show_spectrogram(ax_orig, spectral[idx], vmin=vmin, vmax=vmax, title=f"C{cluster_id} Spectrogram", self_norm=True, cmap="Reds", show_axes=True)
    im = show_spectrogram(ax_rec, reconstructed[idx], vmin=vmin, vmax=vmax, title="Reconstructed", self_norm=True, cmap="Reds", show_axes=True)
    ax_rec.text(0.02, 0.94, f"RMS = {representative_rms[cluster_id]:.2e}", transform=ax_rec.transAxes, fontsize=5.6, va="top", ha="left", bbox=dict(facecolor="0.82", edgecolor="none", pad=1.2))
    ax_orig.set_xlabel("")
    ax_orig.set_xticklabels([])
    ax_rec.set_xlabel("Time (s)")
    if col == 0:
        ax_orig.set_ylabel("Frequency (Hz)")
        ax_rec.set_ylabel("Frequency (Hz)")
    else:
        ax_orig.set_ylabel("")
        ax_rec.set_ylabel("")
        ax_orig.set_yticklabels([])
        ax_rec.set_yticklabels([])

spec_axes[0].text(-0.34, 1.28, "e", transform=spec_axes[0].transAxes, fontsize=12, fontweight="bold", va="top", ha="left")
cbar = fig.colorbar(im, ax=spec_axes, orientation="horizontal", fraction=0.022, pad=0.075)
cbar.set_label("Self-normalized spectral amplitude")

output_base = OUT_DIR / f"cluster_result_DEC_quality_all_events_{EVENT}"
fig.savefig(output_base.with_suffix(".png"), dpi=400, bbox_inches="tight")
plt.close(fig)

print("Saved:")
print(output_base.with_suffix(".png"))

## 5. Preview

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT_DIR / f"cluster_result_DEC_quality_all_events_{EVENT}.png")))